# 02 · Network data audit (agaar.gov.mn, Ulaanbaatar, winter 2025/26)

**Goal:** decide which stations enter the network-design experiment and lock the event threshold.

**Needs:** the agaar archive copied from data-mn into `data/raw/agaar/`
(`agaar-stations.csv` and `daily/agaar-daily-YYYY-MM.csv`).
Daily files are used, not hourly: the site deletes hourly concentrations once a month is validated.

Logic: `src/uaq/agaar.py` (loading + cleaning) and `src/uaq/audit_network.py`.

In [ ]:
# Setup: works whether or not you ran `pip install -e .`
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from uaq.config import load_config, ROOT
cfg = load_config()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
%matplotlib inline
print("Project root:", ROOT)

In [ ]:
display(pd.DataFrame(cfg["network"]["split"]).T.rename_axis("split"))
print("Cleaning rules:", cfg["network"]["cleaning"])

## 1 · Load and clean
Same rules as data-mn's own build: drop negative values, PM2.5 above PM10, spikes (>5× that day's cross-station median and >150), and stuck sensors (same value 5+ days).

In [ ]:
from uaq import audit_network as an
r = an.prepare(cfg)
daily, stations = r["daily"], r["stations"]
print("Station-days by cleaning outcome:")
display(daily["dropped_reason"].replace("", "kept").value_counts().to_frame("station-days"))
names = stations.set_index("station_code")["name_mn"]
stations

## 2 · Coverage by split → eligible stations
A station is eligible with ≥75% valid days in train **and** validation **and** test.

In [ ]:
an.plot_coverage(r["coverage"], names, cfg["quality"]["min_valid_frac"]); plt.show()
print(len(r["eligible"]), "eligible:", r["eligible"])

In [ ]:
an.plot_map(stations, r["eligible"]); plt.show()

## 3 · The winter in one picture
City median and the range across eligible stations, with the splits shaded. Note the validation month (January) is the most polluted; the test months (Feb–Mar) are cleaner.

In [ ]:
w = r["wide"]
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.fill_between(w.index, w.min(axis=1), w.max(axis=1), alpha=0.25, label="min–max across stations")
ax.plot(w.index, w.median(axis=1), lw=1.2, label="median station")
ax.axhline(r["threshold"], color="#C44E52", ls="--", lw=1, label=f"threshold {r['threshold']} µg/m³")
for name, s in cfg["network"]["split"].items():
    ax.axvspan(pd.Timestamp(s["start"]), pd.Timestamp(s["end"]), alpha=0.06, color={"train": "C0", "val": "C1", "test": "C3"}[name])
ax.set_ylabel("Daily mean PM2.5 (µg/m³)"); ax.legend(fontsize=8); plt.show()

## 4 · How redundant are the stations?
High correlation between stations is what makes a smaller network possible, and what makes the choice non-trivial: the best set avoids keeping near-duplicates.

In [ ]:
an.plot_correlation(r["wide"], r["masks"]["train"]); plt.show()

**Check before trusting the GA:** a station that correlates weakly with every other one is either a genuinely different micro-environment (worth keeping) or a faulty sensor (should be excluded). Plot any such station against the city median below and decide; record the decision in the report.

In [ ]:
corr = np.log1p(r["wide"][r["masks"]["train"]]).corr()
mean_corr = (corr.sum() - 1) / (len(corr) - 1)
display(mean_corr.sort_values().round(2).to_frame("mean correlation with other stations").head(5))
odd = mean_corr.nsmallest(2).index.tolist()
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(r["wide"].index, r["wide"].median(axis=1), color="k", lw=1.5, label="city median")
for c in odd:
    ax.plot(r["wide"].index, r["wide"][c], lw=1, label=f"{c} {names[c]}")
ax.set_ylabel("PM2.5 (µg/m³)"); ax.legend(fontsize=8); plt.show()

## 5 · Event threshold (training station-days)

In [ ]:
display(r["rates"].round(3))
print(f"Chosen: {r['threshold_name']} = {r['threshold']} µg/m³\n{r['threshold_why']}")

## 6 · Save the audit report

In [ ]:
report = an.run()
print(report.read_text()[:1500])